# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL (Croissant JSON-LD schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will print all available record set `@id`s, their names, and their contained fields and column IDs.

In [ ]:
# List all record sets and their field @ids for reference
record_sets = list(dataset.record_sets)
print(f"Total record sets: {len(record_sets)}\n")

record_sets_summary = []
for rs in record_sets:
    print(f"Record Set @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description}")
    field_ids = [f.id for f in rs.fields]
    print(f"  Fields (@id): {field_ids}")
    # Some record sets may have columns instead of or in addition to fields
    if hasattr(rs, 'columns') and rs.columns:
        print(f"  Columns (@id): {[c.id for c in rs.columns]}")
    print()
    record_sets_summary.append({
        'id': rs.id,
        'name': rs.name,
        'description': rs.description,
        'fields': field_ids,
        'columns': [c.id for c in rs.columns] if hasattr(rs, 'columns') and rs.columns else []
    })

# Example: Print a few sample records from the first record set (if exists)
if record_sets:
    rs_id = record_sets[0].id
    print(f"\nSample records from record set with @id '{rs_id}':")
    for i, rec in enumerate(dataset.records(record_set=rs_id)):
        print(f"  {rec}")
        if i > 2:
            break

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis.

You can use the summary above to choose the desired record sets by their `@id` values. We'll extract all of them here, indexed by their `@id`.

In [ ]:
# Collect data from all record sets into DataFrames
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Show the first record set's columns and head
if record_set_ids:
    first_rs_id = record_set_ids[0]
    print(f"Columns in record set {first_rs_id}:\n{dataframes[first_rs_id].columns.tolist()}")
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Let's assume the main survey responses are in the primary record set. We'll look for sample numeric fields such as scores (e.g., 'phq9_score', 'gad7_score', or 'psq_score') and group by a categorical field (e.g., 'gender', 'level_of_education', etc.).

Replace `<numeric_field_id>` and `<group_field_id>` below with actual field names as printed above.

In [ ]:
# Choose the main survey record set (likely the largest one with most participant responses)
survey_rs_id = record_set_ids[0]  # Replace or adjust if needed after preview above
df = dataframes[survey_rs_id]

# Inspect the available columns
print(f"Columns in '{survey_rs_id}': {df.columns.tolist()}")

# Try to pick a relevant numeric field (e.g., 'phq9_score', 'gad7_score', 'psq_score')
possible_score_fields = [col for col in df.columns if 'score' in col.lower()]
if possible_score_fields:
    numeric_field = possible_score_fields[0]
    print(f"Using numeric field '{numeric_field}' for demo.")
else:
    # Fallback: use first numeric column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            print(f"Using first numeric column '{numeric_field}' for demo.")
            break
    else:
        raise ValueError('No numeric field found for EDA.')

# Try to pick a grouping field (categorical: gender, education, etc.)
candidate_cats = ['gender', 'level_of_education', 'marital_status', 'village']
group_field = None
for field in candidate_cats:
    matches = [col for col in df.columns if field in col.lower()]
    if matches:
        group_field = matches[0]
        print(f"Using group field '{group_field}' for demo.")
        break
if group_field is None:
    print('No group field found, skipping grouping step.')

# -- Filtering, Normalizing, Grouping --
threshold = df[numeric_field].mean() + df[numeric_field].std() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f}: {len(filtered_df)} records")
display(filtered_df[[numeric_field]].head())

# Normalization
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized '{numeric_field}' for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Grouped aggregates
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(numeric_field, ascending=False)
    print(f"Grouped data by '{group_field}', showing mean {numeric_field} by group:")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
if group_field:
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f"Distribution of {numeric_field} by {group_field}")
    plt.xticks(rotation=45)
else:
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
plt.tight_layout()
plt.show()

## 6. Conclusion
In this notebook, you learned how to:
- Load a FAIR² dataset defined by a Croissant schema using `mlcroissant`
- Review available record sets, fields, and their unique `@id`s
- Extract data in DataFrames for analysis
- Perform exploratory data analysis, including normalization and group-based summarization
- Visualize survey data by demographic attributes

This workflow enables reliable, reproducible FAIR analysis of AI-ready data structures using Croissant.